# Shehab Individual Modelling Notebook

## Purpose

This notebook develops Logistic Regression and Random Forest models for credit card fraud detection using the shared processed datasets created by the group preprocessing notebook.

## Import Libraries

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Load Helper Functions from src

In [ ]:
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import DATA_PROCESSED_DIR, OUTPUT_FIGURES_DIR, OUTPUT_RESULTS_DIR, RANDOM_STATE, TARGET_COLUMN
from src.evaluation import calculate_metrics, create_classification_report
from src.plotting import plot_confusion_matrix, plot_precision_recall_curve, plot_roc_curve

## Configuration

In [ ]:
TRAIN_PATH = DATA_PROCESSED_DIR / "train_processed.csv"
VAL_PATH = DATA_PROCESSED_DIR / "val_processed.csv"
TEST_PATH = DATA_PROCESSED_DIR / "test_processed.csv"

OUTPUT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Load Pre-Processed Datasets

Run the group preprocessing notebook first so these files exist.

In [ ]:
missing_files = [path for path in [TRAIN_PATH, VAL_PATH, TEST_PATH] if not path.exists()]
if missing_files:
    missing_names = ", ".join(str(path.relative_to(PROJECT_ROOT)) for path in missing_files)
    raise FileNotFoundError(f"Processed dataset files are missing: {missing_names}")

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

display(train_df.head())
print(train_df.shape, val_df.shape, test_df.shape)

## Define X and y

In [ ]:
for split_name, split_df in {"train": train_df, "validation": val_df, "test": test_df}.items():
    if TARGET_COLUMN not in split_df.columns:
        raise ValueError(f"{split_name} data does not contain the target column '{TARGET_COLUMN}'.")

X_train = train_df.drop(columns=[TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN]

X_val = val_df.drop(columns=[TARGET_COLUMN])
y_val = val_df[TARGET_COLUMN]

X_test = test_df.drop(columns=[TARGET_COLUMN])
y_test = test_df[TARGET_COLUMN]

print(X_train.shape, X_val.shape, X_test.shape)

## Evaluation Helper

In [ ]:
def positive_class_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return None


def evaluate_model(model, model_name, X, y, dataset_split, notes=""):
    y_pred = model.predict(X)
    y_score = positive_class_scores(model, X)
    metrics = calculate_metrics(y, y_pred, y_score)
    return {
        "model_name": model_name,
        "dataset_split": dataset_split,
        **metrics,
        "notes": notes,
    }

## Baseline Logistic Regression

In [ ]:
baseline_log_reg = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

baseline_log_reg.fit(X_train, y_train)

## Tuned Logistic Regression using GridSearchCV

In [ ]:
log_reg_grid = {
    "model__C": [0.1, 1.0, 10.0],
    "model__solver": ["liblinear"],
}

tuned_log_reg_search = GridSearchCV(
    estimator=Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
        ]
    ),
    param_grid=log_reg_grid,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
)

tuned_log_reg_search.fit(X_train, y_train)
tuned_log_reg = tuned_log_reg_search.best_estimator_
print(tuned_log_reg_search.best_params_)

## Baseline Random Forest

In [ ]:
baseline_random_forest = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

baseline_random_forest.fit(X_train, y_train)

## Tuned Random Forest using RandomizedSearchCV

In [ ]:
random_forest_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

tuned_random_forest_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    param_distributions=random_forest_params,
    n_iter=10,
    scoring="average_precision",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

tuned_random_forest_search.fit(X_train, y_train)
tuned_random_forest = tuned_random_forest_search.best_estimator_
print(tuned_random_forest_search.best_params_)

## Model Evaluation on Validation Data

In [ ]:
models = {
    "Logistic Regression Baseline": baseline_log_reg,
    "Logistic Regression Tuned": tuned_log_reg,
    "Random Forest Baseline": baseline_random_forest,
    "Random Forest Tuned": tuned_random_forest,
}

validation_results = [
    evaluate_model(model, model_name, X_val, y_val, "validation")
    for model_name, model in models.items()
]

validation_results_df = pd.DataFrame(validation_results)
display(validation_results_df.sort_values("pr_auc", ascending=False))

## Select Model for Final Test Evaluation

Select the final model using validation results only. The test set is reserved for final evaluation.

In [ ]:
best_validation_row = validation_results_df.sort_values("pr_auc", ascending=False).iloc[0]
best_model_name = best_validation_row["model_name"]
best_model = models[best_model_name]

print(f"Selected model based on validation PR-AUC: {best_model_name}")

## Final Evaluation on Test Data

In [ ]:
test_result = evaluate_model(
    best_model,
    best_model_name,
    X_test,
    y_test,
    "test",
    notes="Final test evaluation after validation-based model selection",
)

test_results_df = pd.DataFrame([test_result])
display(test_results_df)

## Classification Report

In [ ]:
test_predictions = best_model.predict(X_test)
print(create_classification_report(y_test, test_predictions))

## Confusion Matrix, ROC Curve, and Precision-Recall Curve

In [ ]:
test_scores = positive_class_scores(best_model, X_test)

fig, ax = plot_confusion_matrix(y_test, test_predictions, title=f"{best_model_name} - Test Confusion Matrix")
fig.savefig(OUTPUT_FIGURES_DIR / "shehab_confusion_matrix.png", dpi=150)

if test_scores is not None and y_test.nunique() > 1:
    fig, ax = plot_roc_curve(y_test, test_scores, title=f"{best_model_name} - Test ROC Curve")
    fig.savefig(OUTPUT_FIGURES_DIR / "shehab_roc_curve.png", dpi=150)

    fig, ax = plot_precision_recall_curve(y_test, test_scores, title=f"{best_model_name} - Test Precision-Recall Curve")
    fig.savefig(OUTPUT_FIGURES_DIR / "shehab_precision_recall_curve.png", dpi=150)

## Results Table

In [ ]:
results_df = pd.concat([validation_results_df, test_results_df], ignore_index=True)
results_df = results_df[
    [
        "model_name",
        "dataset_split",
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "roc_auc",
        "pr_auc",
        "notes",
    ]
]

display(results_df)
results_df.to_csv(OUTPUT_RESULTS_DIR / "results_shehab.csv", index=False)

## Interpretation Placeholders

### Which Model Performed Better?

Add the comparison after running the notebook with the real processed datasets.

### Why Recall and PR-AUC Matter for Fraud Detection

Recall matters because missed fraud can create direct financial loss. PR-AUC is useful when fraud cases are rare because it focuses on performance for the positive class.

### What This Means for a Fraud Risk Manager

Explain how the selected model could support review queues, step-up checks, or blocking decisions after results are available.

### Limitations and Next Steps

Record limitations linked to the dataset, modelling choices, threshold selection, and operational deployment.